In [1]:
import re
from pathlib import Path

In [2]:
# 1. Manage footnotes

# Split footnotes from main body and parse them
def parse_footnotes(text: str) -> dict[int, str]:
    """
    Parses footnote blocks of the form:
    [1] Some text that may
    wrap across lines.

    [2] Next footnote...
    """
    # Split on a footnote marker that starts a line: [123]
    # Keep the marker with each subsequent chunk via lookahead split.
    pattern = re.compile(r'^\[(\d+)\]\s*', re.MULTILINE)

    matches = list(pattern.finditer(text))
    footnotes = {}

    for i, m in enumerate(matches):
        fid = int(m.group(1))
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        raw_body = text[start:end]

        # Collapse internal line-wraps into single spaces, normalize whitespace
        body = re.sub(r'\s+', ' ', raw_body).strip()
        footnotes[fid] = body

    return footnotes

# Use structure {footnote_vol: {footenote_id: text}}
footnotes_dict = {}
for vol_num in range(1,7):
    footnotes_path = Path(f"Data/Footnotes Vol {vol_num}.txt")
    footnotes_text = footnotes_path.read_text(encoding="utf-8")
    footnotes_dict[f"footnotes_vol{vol_num}"] = parse_footnotes(footnotes_text)

In [3]:
# Parsing and structuring each volume
'''
Structure example:
{
"volume": 1,
"book": "II",
"chapter": 13,
"chapter_alt_num": 15,
"title": Found on same line as chapter,
"text": "..."
}
'''
vol1_text = Path("Data/Natural History Vol 1.txt").read_text(encoding="utf-8")
vol2_text = Path("Data/Natural History Vol 2.txt").read_text(encoding="utf-8")
vol3_text = Path("Data/Natural History Vol 3.txt").read_text(encoding="utf-8")
vol4_text = Path("Data/Natural History Vol 4.txt").read_text(encoding="utf-8")
vol5_text = Path("Data/Natural History Vol 5.txt").read_text(encoding="utf-8")
vol6_text = Path("Data/Natural History Vol 6.txt").read_text(encoding="utf-8")

def parse_volume(vol_num: int, book_text: str, starting_book: int):
    chapter_pattern = re.compile(
        r'^CHAP\.\s+(\d+)\.?\s*(?:\((\d+)\.\)\s*)?[—-]\s*'  # "CHAP. 5. (7.)—"
        r'(.+?)'                                             # title (lazy)
        r'\n\s*\n',                                           # up to blank line
        re.MULTILINE | re.DOTALL
    )
    
    # Find books
    parsed_vol = []
    for m in chapter_pattern.finditer(book_text):
        parsed_vol.append({
            "volume": vol_num,
            "book": -1,
            "chapter": int(m.group(1)),
            "chapter_alt_num": int(m.group(2)) if m.group(2) else None,
            "title": re.sub(r'\s+', ' ', m.group(3)).strip().rstrip('.'),
            "header_start": m.start(),
            "body_start": m.end(),
        })

    # Split books
    cur_book = starting_book
    for i, chap in enumerate(parsed_vol):
        cur_chapter = chap['chapter']
        if i != 0 and cur_chapter < parsed_vol[i-1]['chapter']:
            cur_book += 1
        
        # Update book number
        chap['book'] = cur_book

        # Replace header_start and body_start with text
        end = parsed_vol[i + 1]["header_start"] if i + 1 < len(parsed_vol) else len(book_text)
        text_body = book_text[chap["body_start"]:end]
        chap["text"] = re.sub(r'\n', ' ', text_body).strip()
        del chap["header_start"], chap["body_start"]
    return parsed_vol

# Parsing each volume
parsed_vol1 = parse_volume(1, vol1_text, 2)
parsed_vol2 = parse_volume(2, vol2_text, 6)
parsed_vol3 = parse_volume(3, vol3_text, 11)
parsed_vol4 = parse_volume(4, vol4_text, 18)
parsed_vol5 = parse_volume(5, vol5_text, 24)
parsed_vol6 = parse_volume(6, vol6_text, 32)

# ==================================================================================
# Handling inline footnote markers
footnote_marker_pattern = re.compile(r'\[(\d+)\]')

def strip_footnote_markers(chapter_text: str) -> str:
    """Removes [N] markers from body text before embedding."""
    cleaned = footnote_marker_pattern.sub('', chapter_text)
    # collapse any double-spaces left behind, e.g. "world[86]." -> "world."
    cleaned = re.sub(r'[ \t]+', ' ', cleaned)
    return cleaned.strip()

def extract_footnote_refs(chapter_text: str, context_chars: int = 120):
    refs = []
    for m in footnote_marker_pattern.finditer(chapter_text):
        fid = int(m.group(1))
        start_ctx = max(0, m.start() - context_chars)
        end_ctx = min(len(chapter_text), m.end() + context_chars)
        context = chapter_text[start_ctx:end_ctx]
        context = footnote_marker_pattern.sub('', context)   # drop neighbouring markers
        context = re.sub(r'\s+', ' ', context).strip()
        refs.append({"footnote_id": fid, "context": context})
    return refs

parsed_vols = {1: parsed_vol1, 2: parsed_vol2, 3: parsed_vol3,
               4: parsed_vol4, 5: parsed_vol5, 6: parsed_vol6}
refs_by_vol = {}

for vol, parsed in parsed_vols.items():
    refs = []
    for chap in parsed:
        for r in extract_footnote_refs(chap["text"]):
            r["book"] = chap["book"]
            r["chapter"] = chap["chapter"]
            refs.append(r)
        chap["text"] = strip_footnote_markers(chap["text"])
    refs_by_vol[vol] = refs

print("Sample Parsed Volume")
print(parsed_vols[1])

print("\nSample Footnote Refs")
print(refs_by_vol[1])

Sample Parsed Volume
[{'volume': 1, 'book': 2, 'chapter': 1, 'chapter_alt_num': 1, 'title': 'WHETHER THE WORLD BE FINITE, AND WHETHER THERE BE MORE THAN ONE WORLD', 'text': 'The world, and whatever that be which we otherwise call the heavens, by the vault of which all things are enclosed, we must conceive to be a Deity, to be eternal, without bounds, neither created, nor subject, at any time, to destruction. To inquire what is beyond it is no concern of man, nor can the human mind form any conjecture respecting it. It is sacred, eternal, and without bounds, all in all; indeed including everything in itself; finite, yet like what is infinite; the most certain of all things, yet like what is uncertain, externally and internally embracing all things in itself; it is the work of nature, and itself constitutes nature. It is madness to harass the mind, as some have done, with attempts to measure the world, and to publish these attempts; or, like others, to argue from what they have made out,

In [4]:
# Building annotation records
def build_annotation_records(volume: int, parsed_volume: list, refs: list, footnotes_dict: dict):
    """
    refs: [{footnote_id, context, book, chapter}] for the whole volume
    footnotes_dict: {"footnotes_volN": {id: text}}
    """
    vol_footnotes = footnotes_dict[f"footnotes_vol{volume}"]
    valid_chapters = {(c["book"], c["chapter"]) for c in parsed_volume}

    annotation_records = []
    missing_text, bad_chapter = [], []
    referenced_ids = set()

    for r in refs:
        fid = r["footnote_id"]
        note = vol_footnotes.get(fid)

        if note is None:                      # marker in body, no footnote in file
            missing_text.append(fid)
            continue
        if (r["book"], r["chapter"]) not in valid_chapters:
            bad_chapter.append(fid)           # shouldn't happen; sanity check
            continue

        referenced_ids.add(fid)
        annotation_records.append({
            "volume": volume,
            "book": r["book"],
            "chapter": r["chapter"],
            "footnote_id": fid,
            "cites_context": r["context"],
            "text": note,
        })

    # Footnotes that exist in the file but were never cited inside a chapter body
    orphans = sorted(set(vol_footnotes) - referenced_ids)

    print(f"Vol {volume}: {len(annotation_records)} annotations | "
          f"{len(missing_text)} markers w/o footnote text | "
          f"{len(orphans)} footnotes never cited in a chapter")
    if missing_text: print("  missing text ids:", missing_text[:20])
    if orphans:      print("  orphan ids:", orphans[:20])

    return annotation_records

annotations = []
for vol in range(1, 7):
    annotations.extend(build_annotation_records(vol, parsed_vols[vol], refs_by_vol[vol], footnotes_dict))


Vol 1: 4256 annotations | 0 markers w/o footnote text | 169 footnotes never cited in a chapter
  orphan ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Vol 2: 3128 annotations | 0 markers w/o footnote text | 79 footnotes never cited in a chapter
  orphan ids: [970, 1482, 1699, 1773, 1799, 1801, 1858, 1872, 1888, 1890, 1916, 1940, 1965, 1971, 1984, 2060, 2290, 2747, 3146, 3147]
Vol 3: 3261 annotations | 0 markers w/o footnote text | 58 footnotes never cited in a chapter
  orphan ids: [1, 2, 111, 552, 575, 831, 977, 3267, 3268, 3269, 3270, 3271, 3272, 3273, 3274, 3275, 3276, 3277, 3278, 3279]
Vol 4: 3229 annotations | 0 markers w/o footnote text | 60 footnotes never cited in a chapter
  orphan ids: [3225, 3226, 3227, 3228, 3229, 3230, 3231, 3232, 3233, 3234, 3235, 3236, 3237, 3238, 3239, 3240, 3241, 3242, 3243, 3244]
Vol 5: 3235 annotations | 0 markers w/o footnote text | 33 footnotes never cited in a chapter
  orphan ids: [3231, 3232, 3233, 3234, 3235, 3236,

In [5]:
# Chunking chapters

# Abbreviations that precede a lowercase/numeral continuation in this text and
# should NOT be treated as sentence-ending, even though they end in '.'
no_split_abbrev = re.compile(r'\b(?:B|c|i|e|No|vol|p|pp|&c)\.$')

def split_sentences(text: str) -> list[str]:
    """Splits on sentence-ending punctuation, but refuses to break after
    abbreviations like 'B.' or 'c.' (as in 'B. ix. c. 58')."""
    raw_splits = re.split(r'(?<=[.!?])\s+(?=[A-Z"\u201c])', text)
    sentences, buf = [], ""
    for part in raw_splits:
        buf = f"{buf} {part}".strip() if buf else part
        if no_split_abbrev.search(buf):
            continue  # keep accumulating, this wasn't really a sentence end
        sentences.append(buf)
        buf = ""
    if buf:
        sentences.append(buf)
    return sentences


def chunk_sentences(sentences: list[str], target_words: int = 250, overlap_sentences: int = 1) -> list[str]:
    """Greedily packs sentences into ~target_words chunks, carrying the
    last `overlap_sentences` of each chunk into the start of the next."""
    chunks = []
    current, current_words = [], 0

    for sentence in sentences:
        sentence_words = len(sentence.split())
        if current and current_words + sentence_words > target_words:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            current_words = sum(len(s.split()) for s in current)
        current.append(sentence)
        current_words += sentence_words

    if current:
        chunks.append(" ".join(current))
    return chunks


def enforce_max_words(text: str, max_words: int = 350) -> list[str]:
    """
    Fallback splitter for a chunk that's already over max_words even
    after sentence-level packing (typically one long run-on sentence).
    Tries semicolon clauses first; falls back to hard word-count cuts
    only if a single clause alone exceeds max_words.
    """
    if len(text.split()) <= max_words:
        return [text]              # already fine — nothing to do

    # split on semicolons, keeping the ';' attached to the preceding clause
    clauses = re.split(r'(?<=;)\s+', text)

    pieces, current, current_words = [], [], 0
    for clause in clauses:
        clause_words = len(clause.split())

        if clause_words > max_words:
            # last resort: even one clause alone is too long — hard-cut by words
            if current:
                pieces.append(" ".join(current))
                current, current_words = [], 0
            words = clause.split()
            for i in range(0, len(words), max_words):
                pieces.append(" ".join(words[i:i + max_words]))
            continue

        if current and current_words + clause_words > max_words:
            pieces.append(" ".join(current))   # close out current piece
            current, current_words = [], 0

        current.append(clause)
        current_words += clause_words

    if current:
        pieces.append(" ".join(current))

    return pieces


def sub_chunk_chapters(parsed_volume: list, target_words: int = 250,
                        overlap_sentences: int = 1, max_words: int = 350) -> list[dict]:
    """
    max_words: hard cap — any packed chunk still over this (typically from
    one long run-on sentence) gets further split on semicolon clauses.
    """
    chunk_records = []

    for chap in parsed_volume:
        sentences = split_sentences(chap["text"])
        packed = chunk_sentences(sentences, target_words, overlap_sentences)

        # enforce the hard cap, flattening any oversized piece into multiple
        pieces = []
        for p in packed:
            pieces.extend(enforce_max_words(p, max_words))

        for i, piece in enumerate(pieces, start=1):
            chunk_records.append({
                "type": "chapter",
                "volume": chap["volume"],
                "book": chap["book"],
                "chapter": chap["chapter"],
                "chapter_alt_num": chap["chapter_alt_num"],
                "title": chap["title"],
                "part": i,
                "total_parts": len(pieces),
                "text": piece,
            })

    return chunk_records

all_chunks = []
for vol, parsed in parsed_vols.items():
    all_chunks.extend(sub_chunk_chapters(parsed))

print(f"{len(all_chunks)} chunks total")
lengths = [len(c["text"].split()) for c in all_chunks]
print(f"avg words/chunk: {sum(lengths)/len(lengths):.0f}, max: {max(lengths)}")
single_part = sum(1 for c in all_chunks if c["total_parts"] == 1)
print(f"{single_part} chapters needed no splitting ({single_part/len(all_chunks)*100:.0f}%)")

5134 chunks total
avg words/chunk: 181, max: 350
1487 chapters needed no splitting (29%)
